In [1]:
import pandas as pd
from difflib import SequenceMatcher

In [2]:
# Add the csv files you want to compare here
# Add the csv files you want to compare here
file1 = './Learning_outcomes.csv'
file2 = './Learning_outcomes2.csv'
output_file = './learning_outcome_differences.csv'

In [3]:
# Function to compare two lists of learning outcomes and find differences
def compare_learning_outcomes(outcomes1, outcomes2):
    # Convert lists to sets for easy comparison
    set1 = set(outcomes1)
    set2 = set(outcomes2)

    # Find differences between the two sets
    only_in_first_file = set1 - set2
    only_in_second_file = set2 - set1

    return list(only_in_first_file), list(only_in_second_file)

# Function to compare the two CSV files and write the differences
def compare_csv_files(file1, file2, output_file):
    # Read the CSV files into DataFrames
    df1 = pd.read_csv(file1)
    df2 = pd.read_csv(file2)

    # Group by school, study program, and learning outcome type, and collect learning outcomes into lists
    df1_grouped = df1.groupby(['Skole', 'Studie program', 'Læringsutbytte type'])['Læringsutbytte'].apply(list).reset_index()
    df2_grouped = df2.groupby(['Skole', 'Studie program', 'Læringsutbytte type'])['Læringsutbytte'].apply(list).reset_index()

    # Initialize an empty list to store the differences
    differences = []

    # Iterate over the rows of the first DataFrame
    for index, row in df1_grouped.iterrows():
        school = row['Skole']
        study_program = row['Studie program']
        type_of_learning_outcome = row['Læringsutbytte type']
        learning_outcomes_1 = row['Læringsutbytte']

        # Check if the corresponding group exists in the second DataFrame
        matching_row = df2_grouped[(df2_grouped['Skole'] == school) & 
                                   (df2_grouped['Studie program'] == study_program) & 
                                   (df2_grouped['Læringsutbytte type'] == type_of_learning_outcome)]

        if not matching_row.empty:
            learning_outcomes_2 = matching_row.iloc[0]['Læringsutbytte']  # Extract the list of learning outcomes

            # Compare the learning outcomes between the two files
            only_in_first_file, only_in_second_file = compare_learning_outcomes(learning_outcomes_1, learning_outcomes_2)

            # If there are differences, append to the differences list
            if only_in_first_file or only_in_second_file:
                if only_in_first_file:
                    for outcome in only_in_first_file:
                        differences.append({
                            'Skole': school,
                            'Studie program': study_program,
                            'Læringsutbytte type': type_of_learning_outcome,
                            'Læringsutbytte': outcome,
                            'File': 'File 1'
                        })
                if only_in_second_file:
                    for outcome in only_in_second_file:
                        differences.append({
                            'Skole': school,
                            'Studie program': study_program,
                            'Læringsutbytte type': type_of_learning_outcome,
                            'Læringsutbytte': outcome,
                            'File': 'File 2'
                        })
        else:
            # If the row exists in the first file but not in the second, it's a missing row
            for outcome in learning_outcomes_1:
                differences.append({
                    'Skole': school,
                    'Studie program': study_program,
                    'Læringsutbytte type': type_of_learning_outcome,
                    'Læringsutbytte': outcome,
                    'File': 'File 1'
                })

    # Iterate over the rows of the second DataFrame to find rows missing in the first file
    for index, row in df2_grouped.iterrows():
        school = row['Skole']
        study_program = row['Studie program']
        type_of_learning_outcome = row['Læringsutbytte type']
        learning_outcomes_2 = row['Læringsutbytte']

        # Check if the corresponding group exists in the first DataFrame
        matching_row = df1_grouped[(df1_grouped['Skole'] == school) & 
                                   (df1_grouped['Studie program'] == study_program) & 
                                   (df1_grouped['Læringsutbytte type'] == type_of_learning_outcome)]

        if matching_row.empty:
            # If the row exists in the second file but not in the first, it's a missing row
            for outcome in learning_outcomes_2:
                differences.append({
                    'Skole': school,
                    'Studie program': study_program,
                    'Læringsutbytte type': type_of_learning_outcome,
                    'Læringsutbytte': outcome,
                    'File': 'File 2'
                })

    # Check if no differences were found
    if not differences:
        print("No differences found between the files.")
    else:
        # Create a DataFrame from the differences
        diff_df = pd.DataFrame(differences)

        # Write the differences to a new CSV file
        diff_df.to_csv(output_file, index=False)
        print(f"Differences have been written to {output_file}")

# Compare the files and write the differences to a new CSV file
compare_csv_files(file1, file2, output_file)


Differences have been written to ./learning_outcome_differences.csv
